# 06 - Merge, Join e Concat

## Objetivo
Combinar DataFrames de diferentes fontes.

## Conceitos

### Tipos de juncao (como em SQL)
- `inner`: apenas chaves presentes em ambos.
- `left`: todas as chaves do DataFrame da esquerda.
- `right`: todas as chaves do DataFrame da direita.
- `outer`: todas as chaves de ambos.
- `cross`: produto cartesiano.

### pd.merge
Une dois DataFrames por colunas-chave.
Parametros principais:
- `on`: nome da coluna em comum.
- `left_on`, `right_on`: nomes diferentes.
- `how`: tipo de juncao.
- `suffixes`: sufixos para colunas com mesmo nome.
- `indicator`: adiciona coluna indicando origem da linha.

### pd.concat
Empilha DataFrames:
- `axis=0`: empilha linhas.
- `axis=1`: empilha colunas.
- `ignore_index=True`: reindexa.
- `keys`: cria indice hierarquico.

### df.join
Atalho para merge usando o indice.

### Duplicatas apos merge
Juncoes podem gerar duplicatas se as chaves nao forem unicas.
Use `validate=` para checar cardinalidade.

## DataFrames de exemplo
Dois DataFrames relacionados pela coluna `id_depto`: um com funcionarios
e outro com departamentos. Vamos usar `pd.merge` para combina-los de
diferentes formas.

In [ ]:
import pandas as pd

# DataFrames de exemplo
funcionarios = pd.DataFrame({
    "id_func": [1, 2, 3, 4, 5],
    "nome": ["Ana", "Bruno", "Carla", "Diego", "Elisa"],
    "id_depto": [10, 20, 10, 30, 20],
})

departamentos = pd.DataFrame({
    "id_depto": [10, 20, 40],
    "nome_depto": ["TI", "RH", "Vendas"],
})

print("Funcionarios:\n", funcionarios)
print("\nDepartamentos:\n", departamentos)

## Tipos de juncao com merge
- **inner**: apenas chaves presentes nos dois lados.
- **left**: todas as chaves da esquerda (funcionarios); chaves sem par viram NaN.
- **right**: todas as chaves da direita (departamentos).
- **outer**: uniao das chaves; NaN onde nao ha correspondencia.

In [ ]:
# Inner merge
inner = pd.merge(funcionarios, departamentos,
                 on="id_depto", how="inner")
print("\nInner:\n", inner)

# Left merge
left = pd.merge(funcionarios, departamentos,
                on="id_depto", how="left")
print("\nLeft:\n", left)

# Right merge
right = pd.merge(funcionarios, departamentos,
                 on="id_depto", how="right")
print("\nRight:\n", right)

# Outer merge
outer = pd.merge(funcionarios, departamentos,
                 on="id_depto", how="outer")
print("\nOuter:\n", outer)

## indicator
`indicator=True` adiciona uma coluna `_merge` que mostra a origem de
cada linha: `both`, `left_only` ou `right_only`. Muito util para
auditar o resultado de uma juncao.

In [ ]:
# Indicador de origem
ind = pd.merge(funcionarios, departamentos,
               on="id_depto", how="outer", indicator=True)
print("\nCom indicator:\n", ind)

## Chaves com nomes diferentes
Quando as colunas-chave tem nomes diferentes nos dois DataFrames,
usamos `left_on` e `right_on` em vez de `on`.

In [ ]:
# Colunas com nomes diferentes
dep2 = departamentos.rename(columns={"id_depto": "codigo"})
merge_diff = pd.merge(funcionarios, dep2,
                      left_on="id_depto", right_on="codigo",
                      how="left")
print("\nleft_on / right_on:\n", merge_diff)

## suffixes
Quando ambos os DataFrames tem colunas com o mesmo nome (alem da chave),
`suffixes=("_a", "_b")` diferencia as colunas no resultado.

In [ ]:
# Colunas homonimas com suffixes
a = pd.DataFrame({"id": [1, 2], "valor": [10, 20]})
b = pd.DataFrame({"id": [1, 2], "valor": [100, 200]})
merged = pd.merge(a, b, on="id", suffixes=("_a", "_b"))
print("\nSuffixes:\n", merged)

## validate
`validate=` checa a cardinalidade da juncao:
- `"one_to_one"`: chave unica em ambos.
- `"one_to_many"`: unica a esquerda, duplicada a direita.
- `"many_to_one"`: duplicada a esquerda, unica a direita.
- `"many_to_many"`: duplicada em ambos.

Se a cardinalidade nao for a esperada, o Pandas levanta erro.

In [ ]:
# Validando cardinalidade
try:
    pd.merge(funcionarios, departamentos,
             on="id_depto", validate="one_to_one")
except Exception as e:
    print("\nErro de validacao:", e)

## pd.concat — empilhamento
- `axis=0`: empilha **linhas** (equivalente a UNION ALL em SQL).
- `axis=1`: empilha **colunas** (equivalente a juntar lado a lado).
- `ignore_index=True`: reindexa as linhas de 0 a N-1.
- `keys`: cria um indice hierarquico identificando a origem.

In [ ]:
# concat linhas
df1 = pd.DataFrame({"x": [1, 2], "y": [3, 4]})
df2 = pd.DataFrame({"x": [5, 6], "y": [7, 8]})
print("\nconcat axis=0:\n", pd.concat([df1, df2], ignore_index=True))

# concat colunas
print("\nconcat axis=1:\n", pd.concat([df1, df2],
                                       axis=1,
                                       keys=["a", "b"]))

## df.join — merge pelo indice
`df.join(outro)` e um atalho para merge usando o **indice** como chave.
Aceita `how` com os mesmos valores de `pd.merge`.

In [ ]:
# join pelo indice
esq = pd.DataFrame({"valor": [1, 2, 3]}, index=["a", "b", "c"])
dir_ = pd.DataFrame({"outro": [10, 20, 30]}, index=["a", "b", "d"])
print("\njoin:\n", esq.join(dir_, how="outer"))

## cross merge
`how="cross"` produz o **produto cartesiano**: cada linha de um
DataFrame combinada com cada linha do outro. Nao precisa de chave comum.

In [ ]:
# cross merge
c = pd.merge(pd.DataFrame({"x": [1, 2]}),
             pd.DataFrame({"y": ["a", "b"]}),
             how="cross")
print("\ncross:\n", c)